In [ ]:
#r "FreeXNSE.dll"
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using FreeXNSE;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

In [ ]:
string proj = "ContactAngleHysteresis";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [ ]:
var sessions = BoSSSshell.wmg.Sessions;
sessions

In [ ]:
//Directory.CreateDirectory(BoSSSshell.wmg.CurrentProject);
//sessions.ForEach(s => File.Copy(s.Database.Path + "\\sessions\\" + s.ID + "\\ContactAngle.txt", Path.Combine(BoSSSshell.wmg.CurrentProject, s.Name) + ".txt", true));

In [ ]:
var sess = sessions.Where(s => 
                          s.Name == "DropletContactAngleHysteresis_BndySlip0.0_0.0_LineSlip0.0_1.0_Visc1.0_0.0_CAeq1.5708_CAa3.1416_CAr1.3090"
                         || s.Name == "DropletContactAngleHysteresis_BndySlip0.0_0.0_LineSlip0.0_1.0_Visc1.0_0.0_CAeq1.5708_CAa1.8326_CAr0.0000"
                         );
//sess.ForEach(s => s.Timesteps.Where(ts => ts.TimeStepNumber.Length == 1 && (ts.TimeStepNumber == 300 ||ts.TimeStepNumber == 0)).Export().To(Path.GetFullPath(Path.Combine(s.ProjectName, s.Name))).WithSupersampling(4).WithShadowFields().Do());

In [ ]:
//sessions.ElementAt(21).Export().To(Path.GetFullPath(Path.Combine(sessions.ElementAt(21).ProjectName, sessions.ElementAt(21).Name))).WithSupersampling(2).WithShadowFields().Do();
//sessions.ElementAt(18).Export().To(Path.GetFullPath(Path.Combine(sessions.ElementAt(18).ProjectName, sessions.ElementAt(18).Name))).WithSupersampling(2).WithShadowFields().Do();

Assert that the moving contact line assumes a contact angle matching the specified advancing or receding value  
Assert that the pinned contact line assumes a contact angle according to  
$$
\cos(\theta_{adv})-\cos(\theta_{rec}) = -\frac{\pi R^2 We}{2 Fr}
$$
(The simualtions are all created using $Fr = \pi/2 R^2 We$, such that $\cos(\theta_{adv})-\cos(\theta_{rec}) = -1$)  
I decided to test here the deviation in the cosines.

In [ ]:
bool success = true;
double thrsh = 0.05;
var SSSS = sessions.Where(s => !s.Name.Contains("BndySlipInfinity")); // Exclude no-slip simulations
var data = SSSS.ReadLogDataForMovingContactLine();

for(int i = 0; i < SSSS.Count(); i++){
    var s = SSSS.ElementAt(i);

    double rec = Convert.ToDouble(s.KeysAndQueries["id:thetaRec"]); // configuration value
    double adv = Convert.ToDouble(s.KeysAndQueries["id:thetaAdv"]); // configuration value
    // Console.WriteLine(s.Name);
    // Console.WriteLine(rec);
    // Console.WriteLine(adv);

    double rec_sim;
    double adv_sim;

    // just load the last value
    if(data[0].ElementAt(1).dataGroups.ElementAt(i).Values.First() > data[1].ElementAt(1).dataGroups.ElementAt(i).Values.First()){ // data[0] is receding
        rec_sim = data[0].ElementAt(5).dataGroups.ElementAt(i).Values.Last() / 180.0 * Math.PI;
        adv_sim = data[1].ElementAt(5).dataGroups.ElementAt(i).Values.Last() / 180.0 * Math.PI;
    } else { // data[0] is advancing
        adv_sim = data[0].ElementAt(5).dataGroups.ElementAt(i).Values.Last() / 180.0 * Math.PI;
        rec_sim = data[1].ElementAt(5).dataGroups.ElementAt(i).Values.Last() / 180.0 * Math.PI;
    }

    // Console.WriteLine(rec_sim);
    // Console.WriteLine(adv_sim);

    double err0 = Math.Abs(Math.Cos(adv_sim) - Math.Cos(rec_sim) + 1);
    double err1;

    if(rec == 0){ // receding contact line is pinned
        err1 = Math.Abs(Math.Cos(adv_sim) - Math.Cos(adv));
        if(err1 > thrsh){
            success &= false;
            Console.WriteLine("Unexpected advancing contact angle in {0}", s.Name);
        }
    } else { // advancing contact line is pinned
        err1 = Math.Abs(Math.Cos(rec_sim) - Math.Cos(rec));
        if(err1 > thrsh){
            success &= false;
            Console.WriteLine("Unexpected advancing contact angle in {0}", s.Name);
        }        
    }

    if(err0 > thrsh){
        success &= false;
        Console.WriteLine("Wrong distance between advancing and receding contact angle in {0}", s.Name);
    } 
}

if(!success){
    throw new ApplicationException("Not all contact angles in expected range!");
}